### Задача 1 (10 баллов). 

Представьте, что вы разрабатываете игру, в которой игрок управляет флотом космических кораблей. Каждый корабль имеет свои характеристики, вооружение и экипаж. Чтобы управлять флотом, игрок должен делегировать различные задачи (например, атака, оборона, ремонт) разным кораблям и экипажу. Ваша задача — создать систему классов, которая использует композицию для организации кораблей, экипажа и вооружения, а также делегирует задачи между этими объектами.

Требования:

Создайте класс Ship, который представляет космический корабль. Корабль должен:

- Иметь имя, тип корабля (например, "battlecruiser", "frigate", "destroyer") и уровень прочности (например, 100%).
- Иметь экипаж, состоящий из пилота и инженера (с помощью композиции).
- Иметь вооружение (также через композицию), которое будет делегироваться классу Weapon.
- Мог выполнять действия атаки, ремонта и защиты, но делегировать эти задачи соответствующим объектам (например, пилоту или инженеру).

Создайте класс CrewMember (член экипажа), который может выполнять различные задачи:

- Pilot — отвечает за управление кораблём, атаки и манёвры.
- Engineer — отвечает за ремонт и восстановление прочности корабля.

Используйте композицию для того, чтобы корабль "имел" пилота и инженера.

Создайте класс Weapon, который будет представлять вооружение корабля:

- Каждый корабль может иметь одно или несколько оружий.
- Оружие может атаковать противника, но уровень атаки зависит от типа оружия и его состояния (например, лазер, ракеты, плазменные пушки).

Используйте композицию для добавления оружия к кораблю и делегируйте задачи атаки объектам класса Weapon.

Класс Fleet должен представлять целый флот кораблей и управлять их действиями:

- Возможность добавлять корабли во флот.
- Делегирование задач атаки и ремонта флоту, который будет распределять их между кораблями.

Пример использования:


#### Определение оружия

    laser = Weapon("Laser Cannon", 50)
    missile = Weapon("Missile Launcher", 100)

#### Создание экипажа

    pilot = Pilot("John Doe")
    engineer = Engineer("Jane Smith")

#### Создание кораблей

    ship1 = Ship("USS Enterprise", "battlecruiser", pilot, engineer)
    ship2 = Ship("Falcon", "frigate", Pilot("Han Solo"), Engineer("Chewbacca"))

#### Добавление вооружения к кораблям

    ship1.add_weapon(laser)
    ship2.add_weapon(missile)

#### Создание флота

    fleet = Fleet()
    fleet.add_ship(ship1)
    fleet.add_ship(ship2)

#### Атака флотом

    print("Флот атакует!")
    fleet.attack_all()

#### Ремонт флота

    print("\nФлот выполняет ремонт!")
    fleet.repair_all()

#### Результат:

    # USS Enterprise атакует с помощью Laser Cannon (урон 50)
    # Falcon атакует с помощью Missile Launcher (урон 100)
    # USS Enterprise был отремонтирован инженером Jane Smith до полной прочности.
    # Falcon был отремонтирован инженером Chewbacca до полной прочности.
    
Подсказки:

Композиция: Класс Ship должен содержать объекты экипажа и вооружения, а класс Fleet должен содержать объекты кораблей.

Делегирование: Методы атаки, защиты и ремонта должны вызывать методы у соответствующих объектов. Например, при вызове метода attack() у корабля, этот метод должен делегировать выполнение атаки объекту Weapon и экипажу (пилоту).

Взаимодействие классов: Корабль не выполняет все задачи сам, он делегирует их своим компонентам (экипажу и оружию).

In [6]:
import random

In [7]:
class CrewMember:
    def __init__(self, name):
        self.name = name


class Pilot(CrewMember):
    def attack(self, ship, target, weapon):
        if not target.alive:
            print(f"{target.name} уже уничтожен")
            return

        # промах зависит от уклонения цели
        if random.random() < target.evasion:
            print(f"{ship.name} промахивается! {target.name} уклонился")
            return

        # вероятность фейла оружия
        if random.random() < weapon.failure_rate:
            print(f"оружие {weapon.name} на {ship.name} отказало!")
            return

        # критический удар
        damage = weapon.damage
        if random.random() < weapon.crit_rate:
            damage *= 2
            print(f"критический удар! урон удвоен до {damage}")

        # броня цели уменьшает урон
        final_damage = max(1, damage - target.armor)
        target.hull -= final_damage
        print(f"{ship.name} атакует {target.name} ({weapon.name}): {final_damage} урона")

        if target.hull <= 0:
            target.alive = False
            target.hull = 0
            print(f"{target.name} уничтожен!")

    def maneuver(self, ship):
        print(f"{ship.name} выполняет манёвр уклонения под управлением пилота {self.name}")


class Engineer(CrewMember):
    def repair(self, ship):
        if not ship.alive:
            print(f"{ship.name} уничтожен. ремонт невозможен")
            return
        
        repair_amount = random.randint(15, 30)
        ship.hull = min(ship.hull + repair_amount, 100)
        print(f"инженер {self.name} ремонтирует {ship.name}: +{repair_amount} к прочности")

    def repair_weapon(self, weapon):
        if random.random() < 0.3:
            weapon.failure_rate *= 0.5
            print(f"инженер {self.name} улучшил надёжность оружия {weapon.name}!")
        else:
            print(f"инженер {self.name} проверил оружие {weapon.name}, всё в норме")

In [8]:
class Weapon:
    def __init__(self, name, damage, crit_rate=0.15, failure_rate=0.1):
        self.name = name
        self.damage = damage
        self.crit_rate = crit_rate
        self.failure_rate = failure_rate

In [9]:
class Ship:
    def __init__(self, name, ship_type, pilot: Pilot, engineer: Engineer):
        self.name = name
        self.ship_type = ship_type
        self.pilot = pilot
        self.engineer = engineer

        self.hull = 100
        self.armor = random.randint(5, 15)
        self.evasion = random.random() * 0.2  
        self.weapons = []
        self.alive = True

    def add_weapon(self, weapon):
        self.weapons.append(weapon)

    def attack(self, target):
        if not self.weapons:
            print(f"{self.name}: нет оружия для атаки.")
            return

        weapon = random.choice(self.weapons)
        self.pilot.attack(self, target, weapon)

    def repair(self):
        self.engineer.repair(self)

    def defend(self):
        self.pilot.maneuver(self)

    def __repr__(self):
        return f"{self.name} (hull={self.hull}, armor={self.armor}, evasion={self.evasion:.2f})"

In [10]:
class Fleet:
    def __init__(self, name="Fleet"):
        self.name = name
        self.ships = []

    def add_ship(self, ship):
        self.ships.append(ship)

    def attack_all(self, target_fleet):
        print(f"\n{self.name} начинает атаку на {target_fleet.name}!")
        for ship in self.ships:
            if ship.alive:
                target = random.choice([s for s in target_fleet.ships if s.alive])
                ship.attack(target)
            else:
                print(f"{ship.name} не может атаковать — уничтожен")

    def repair_all(self):
        print(f"\n{self.name}: начинаем ремонт")
        for ship in self.ships:
            if ship.alive:
                ship.repair()

    def status(self):
        print(f"\nстатус флота {self.name}:")
        for s in self.ships:
            state = "уничтожен" if not s.alive else f"{s.hull}% прочности"
            print(f" - {s.name}: {state}")


In [11]:
laser = Weapon("cool laser", 40)
plasma = Weapon("weird plasma", 55)
missile = Weapon("missile launcher", 70, crit_rate=0.25)

pilot1 = Pilot("luke")
eng1 = Engineer("leia")

pilot2 = Pilot("han solo")
eng2 = Engineer("chewbacca")

ship1 = Ship("death star", "battlecruiser", pilot1, eng1)
ship2 = Ship("life star", "frigate", pilot2, eng2)

ship1.add_weapon(laser)
ship1.add_weapon(plasma)
ship2.add_weapon(missile)

fleetA = Fleet("evolve")
fleetB = Fleet("advance")

fleetA.add_ship(ship1)
fleetB.add_ship(ship2)

fleetA.attack_all(fleetB)
fleetB.attack_all(fleetA)

fleetA.repair_all()
fleetB.repair_all()

fleetA.status()
fleetB.status()


evolve начинает атаку на advance!
death star промахивается! life star уклонился

advance начинает атаку на evolve!
life star атакует death star (missile launcher): 57 урона

evolve: начинаем ремонт
инженер leia ремонтирует death star: +16 к прочности

advance: начинаем ремонт
инженер chewbacca ремонтирует life star: +17 к прочности

статус флота evolve:
 - death star: 59% прочности

статус флота advance:
 - life star: 100% прочности
